## 🏗️ LeetCode 295: Find Median from Data Stream
---

<div style="padding: 15px; border-left: 8px solid #F44336; background-color: #ffebee; color: #b71c1c; border-radius: 4px;">
    <strong>The Core Insight:</strong> Split the data in half. The lower half lives in a max-heap (we want its maximum — the lower median). The upper half lives in a min-heap (we want its minimum — the upper median). Keep the two heaps balanced in size. The median is at the boundary.
</div>

### 🛠️ The Mathematical Model

$$\text{lo} = \text{max-heap (lower half)}, \quad \text{hi} = \text{min-heap (upper half)}$$
$$\text{Invariant: } |\text{lo}| = |\text{hi}| \text{ or } |\text{lo}| = |\text{hi}| + 1$$
$$\text{Median: lo[0] if odd count; } (\text{lo}[0] + \text{hi}[0]) / 2 \text{ if even}$$

---

### 📋 Problem

Implement `MedianFinder` class:
- `addNum(int num)` — adds a number to the data structure
- `findMedian()` — returns the median of all elements so far

**Example 1:**
```
addNum(1) | addNum(2) | findMedian() → 1.5
addNum(3) | findMedian() → 2.0
```

**Constraints:** -10⁵ ≤ num ≤ 10⁵ | At most 5×10⁴ calls to addNum and findMedian | At least one element before findMedian is called

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Two Sorted Stacks</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Sort all values and cut them in half. The left half faces right (you see its top = max of lower half). The right half faces left (you see its top = min of upper half). The median is between these two tops."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Max-heap lo (top = lower median), min-heap hi (top = upper median). Balance sizes. Median at boundary.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Balance Scale</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"A balance scale with two pans. Left pan = lower half, right pan = upper half. They must stay within 1 element of each other. The center weight is the median."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Size invariant: |lo| == |hi| or |lo| == |hi| + 1. Rebalance after each add.</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Sorting the entire list after each addNum() is O(n log n) per call. Two-heap approach: O(log n) per addNum(), O(1) per findMedian().
</div>

## 🐢 Approach 1: Brute Force — $O(n \log n)$ per addNum

In [ ]:
class MedianFinder_Brute:
    """
    Brute Force: Sort on every findMedian call
    addNum: O(1) | findMedian: O(n log n) | Space: O(n)
    """
    def __init__(self):
        self.nums = []

    def addNum(self, num: int) -> None:
        self.nums.append(num)

    def findMedian(self) -> float:
        self.nums.sort()
        n = len(self.nums)
        if n % 2 == 1:
            return float(self.nums[n // 2])
        else:
            return (self.nums[n // 2 - 1] + self.nums[n // 2]) / 2.0


mf = MedianFinder_Brute()
mf.addNum(1); mf.addNum(2)
print(mf.findMedian())   # Expected: 1.5
mf.addNum(3)
print(mf.findMedian())   # Expected: 2.0

## 🔬 Complexity Analysis: $O(n \log n)$ vs. $O(\log n)$ per add

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> The median only depends on two values: the maximum of the lower half and the minimum of the upper half. We don't need the full sorted order. Two heaps maintain exactly these two boundary values in O(log n) per update and O(1) per query.
</div>

---

## 📉 Why Brute Force Fails: The $O(n \log n)$ Trap

Sort on each findMedian() call scales with list size. After m add() calls, findMedian() costs O(m log m) per call.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **50,000 calls** | O(n² log n) total | n sorts, each O(n log n) |
| **High-frequency streaming** | Unusable | Sort cost grows with stream length |

---

## 🚀 The Optimal Approach: $O(\log n)$ per addNum, $O(1)$ per findMedian

Two heaps partition the data:
- `lo`: max-heap (negated in Python) — holds the lower half
- `hi`: min-heap — holds the upper half
- Invariant: `|lo| == |hi|` (even count) or `|lo| == |hi| + 1` (odd count, lo has extra)

### The Key Lifecycle Rule
1. **Always push to lo first** (negate for max-heap)
2. **Rebalance ordering:** if lo's max > hi's min, move lo's top to hi
3. **Rebalance sizes:** if |lo| > |hi| + 1, move lo's top to hi; if |hi| > |lo|, move hi's top to lo

---

## ✅ Mathematical Proof

After every addNum(), the invariant holds: lo contains the lower ⌈n/2⌉ elements, hi contains the upper ⌊n/2⌋. Both heaps give O(1) access to their tops. findMedian() reads lo[0] (and optionally hi[0]) in O(1).

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Two heaps maintain the "boundary" of the sorted data. We never need the full order — just the maximum-of-lower and minimum-of-upper. That's exactly what heaps provide.
</div>

## 🚀 Approach 2: Two Heaps — $O(\log n)$ per addNum, $O(1)$ per findMedian
---

Instead of full sort, we **split the data into two heaps** at the median boundary.

As we iterate:
1. Push to lo (max-heap) first — everything goes through lo
2. Rebalance ordering: ensure lo's max ≤ hi's min (move lo's top to hi if violated)
3. Rebalance sizes: ensure |lo| == |hi| or |lo| == |hi| + 1
4. findMedian(): read lo[0] if odd, average of both tops if even

In [ ]:
import heapq

class MedianFinder:
    """
    Optimal: Two Heaps
    addNum: O(log n) | findMedian: O(1) | Space: O(n)
    """
    def __init__(self):
        self.lo = []   # max-heap (lower half) — store negated values
        self.hi = []   # min-heap (upper half)

    def addNum(self, num: int) -> None:
        heapq.heappush(self.lo, -num)   # always push to lo first (negated)

        # Rebalance ordering: lo's max must be <= hi's min
        if self.lo and self.hi and (-self.lo[0]) > self.hi[0]:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))

        # Rebalance sizes: lo can have at most 1 more element than hi
        if len(self.lo) > len(self.hi) + 1:
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) > len(self.lo):
            heapq.heappush(self.lo, -heapq.heappop(self.hi))

    def findMedian(self) -> float:
        if len(self.lo) > len(self.hi):
            return float(-self.lo[0])   # odd count — median is lo's top
        return (-self.lo[0] + self.hi[0]) / 2.0   # even count — average of both tops


mf = MedianFinder()
mf.addNum(1); mf.addNum(2)
print("Optimal:", mf.findMedian())   # Expected: 1.5
mf.addNum(3)
print("Optimal:", mf.findMedian())   # Expected: 2.0

## 🎤 5 Interview Q&A

### Q1: Why two heaps instead of a sorted list?

**Answer:** Insertion into a sorted list is O(n) — you need to find the correct position and shift elements. Heap insertion (heappush) is O(log n). For streaming data with m addNum() calls, sorted list costs O(m × n) total vs. O(m log n) with heaps. The two-heap approach is also O(1) for findMedian, while sorted list requires no extra computation at query time but pays more at insert time.

---

### Q2: Why always push to lo first, then rebalance?

**Answer:** By routing everything through lo first, we ensure every new element is evaluated against the lower half. If it turns out the element belongs in the upper half (lo's max > hi's min after insert), we move it. This "push-then-check" pattern is simpler than trying to determine the correct heap before pushing.

---

### Q3: What is the time complexity of addNum and findMedian?

**Answer:** addNum = O(log n) — at most 2-3 heap operations (heappush + heappop for rebalancing), each O(log n). findMedian = O(1) — just reads the tops of both heaps without modifying them. Space = O(n) — the two heaps together hold all n elements.

---

### Q4: Where does this pattern apply in real monitoring systems?

**Answer:** Rolling P50 (median) for SLO monitoring. You can extend to rolling P95 by changing the split ratio: lo has 5% of elements, hi has 95% — the P95 is lo's top. In Citi's APM infrastructure, rolling percentile computation over 5-minute windows on response times used this exact pattern: two heaps, rebalanced each minute as new readings arrived and old ones expired.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Single element — lo has 1 element, hi is empty. findMedian returns lo[0]. (2) Two elements — after rebalancing, lo has 1, hi has 1. findMedian averages both tops. (3) Even vs odd count — the size check `len(lo) > len(hi)` (not >=) correctly identifies the odd-count case. (4) Negative numbers — negation for max-heap works correctly: `-(-5) = 5`, `-(-3) = 3`.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Median** | The middle value of a sorted dataset; for even count, average of two middle values |
| **Two-Heap Pattern** | Split data into lower/upper halves using max-heap + min-heap; median is at the boundary |
| **Lower Half** | All elements ≤ median — stored in a max-heap (lo); lo's top = lower median |
| **Upper Half** | All elements ≥ median — stored in a min-heap (hi); hi's top = upper median |
| **Size Invariant** | |lo| == |hi| or |lo| == |hi| + 1 — maintained after every addNum() |
| **Ordering Invariant** | lo's max ≤ hi's min — ensures lo contains only lower-half elements |
| **Rolling Percentile** | Extending two-heap to any percentile p: lo holds p% of elements, hi holds (1-p)% |
| **Streaming Median** | Computing the median of a sequence as elements arrive, without storing all of them sorted |

## 💼 The Citi Narrative

**Context:** Citi's SLO monitoring required tracking P50 (median) API response times in real time for 6,000 monitored services — updated every minute as new requests were logged.

**Scenario:** Each service received ~100 requests per minute. The naive approach: sort 100 response times each minute for each service = 6,000 × O(100 log 100) ≈ 4 million sort operations per minute. Two-heap approach: 6,000 × O(100 × log 100) for add() calls + O(1) for each findMedian() query.

**How this pattern applied:** Each service maintained its own MedianFinder instance. As response times arrived, addNum() updated the two heaps in O(log n). At each minute boundary, findMedian() returned the P50 in O(1) — feeding the SLO dashboard. The pattern extended naturally to P95 by adjusting the size ratio: lo held 5% of responses (max-heap), hi held 95% (min-heap), lo's top = P95.

**Impact:** Real-time P50 and P95 SLO monitoring across 6,000 services without batch sorting. Response time deviation alerts fired within seconds of threshold breach — compared to minute-level delays with the previous sort-based approach.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Extend MedianFinder to track the RUNNING AVERAGE
# alongside the median. Add an average() method.
# -------------------------------------------------------

import heapq

class MedianAndMeanFinder:
    def __init__(self):
        self.lo = []   # max-heap (lower half)
        self.hi = []   # min-heap (upper half)
        self.total = 0
        self.count = 0

    def addNum(self, num: int) -> None:
        # Your solution — same two-heap logic + track total and count
        pass

    def findMedian(self) -> float:
        # Your solution
        pass

    def findMean(self) -> float:
        # Return the running average
        pass


# Test
mf = MedianAndMeanFinder()
for n in [1, 2, 3, 4, 5]:
    mf.addNum(n)
print(mf.findMedian())   # Expected: 3.0
print(mf.findMean())     # Expected: 3.0

## 🎯 Summary: Key Takeaways

### The Pattern
**Two Heaps** — Max-heap for lower half + min-heap for upper half. Median at the boundary.

### When to Use It
- ✅ Streaming median — elements arrive one by one
- ✅ Rolling percentiles (P50, P95) — change the size ratio
- ✅ Need O(1) median query with O(log n) updates
- ❌ **Don't use when:** Batch median on static array — just sort and index

### Complexity
| Approach | addNum | findMedian | Space |
|----------|--------|------------|-------|
| Brute Force (sort) | $O(1)$ | $O(n \log n)$ | $O(n)$ |
| Optimal (Two Heaps) | $O(\log n)$ | $O(1)$ | $O(n)$ |

### Interview Confidence Checklist
- [ ] Can explain why two heaps give O(1) median access
- [ ] Can describe the size invariant and ordering invariant
- [ ] Can write the full MedianFinder from memory in 8 minutes
- [ ] Can extend to arbitrary percentiles (P95, P99)
- [ ] Can connect to SLO monitoring use case with specific numbers

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Two-Heap Median and you've mastered the most elegant data structure trick in streaming algorithms. 🚀
```

---

## B. SQL — Analytical SQL: YoY Growth, Cohort Analysis, Funnel

**Notebook file:** `D:\Workspace\StudyMaterial\Day5\sql-analytical.ipynb`

**Use the SQL section in the existing study plan (marked "## B. SQL — Analytical SQL: YoY Growth, Cohort Analysis, Funnel") for all code cells. Do not invent new code.**

Core concept for notebook: "Analytical SQL separates analysts from engineers. YoY, cohort, and funnel queries appear in every data engineering interview."

Citi narrative: Citi capacity planning team tracked server cohorts by provisioning quarter — "of servers provisioned in Q1 2024, how many are still active in Q4 2025?" The cohort analysis pattern was applied to server lifecycle data instead of user events. Impact: identified decommissioning backlog — 340 servers provisioned but not yet decommissioned, freeing $X in licensing costs.

---

## C. Python — ETL Patterns: argparse, Logging, Pipeline Structure

**Notebook file:** `D:\Workspace\StudyMaterial\Day5\python-etl-patterns.ipynb`

**Use the Python section in the existing study plan (marked "## C. Python — ETL Patterns: argparse, Logging, Pipeline Structure") for all code cells. Do not invent new code.**

Citi narrative: The ETL pipeline template pattern came directly from Citi's capacity data pipeline — `--env`, `--date`, `--dry-run` flags, structured JSON logging, `sys.exit(1)` on failure for Airflow retry detection. Impact: same pipeline code ran in dev, staging, and production with zero code changes — only environment variable differed.

---

## D. Technology — Capacity Planning Methodology

**Notebook file:** `D:\Workspace\StudyMaterial\Day5\capacity-planning.ipynb`

**Use the Tech section in the existing study plan (marked "## D. Technology — Capacity Planning Methodology") for all code cells. Do not invent new code.**

Citi narrative: The Four-Step Loop is Sean's actual methodology from Citi. Baseline: 90-day CA APM data. Model: requests/sec as capacity driver. Forecast: linear extrapolation with 70% ceiling. Result: "We hit ceiling in 47 days" delivered as a Jira ticket 6 weeks before the breach — zero capacity outages in the following 12 months.

---

## Behavioral Anchor — Citi Story #5: Proactive Capacity Planning Win

> **Gemini:** At end of session, say: "Tell me about a time you prevented an outage through proactive capacity planning at Citi. Two minutes — go."

**Story framework (Sean fills in with his details):**
- **Situation:** Monitoring 6,000 endpoints, growing application portfolio
- **Task:** Build a proactive capacity alerting system — no more reactive 3am pages
- **Action:** Built trending model in Python using 90-day APM baselines; set alerts at 60% utilization on the trend line, not the current reading; automated weekly capacity reports to infrastructure teams
- **Result:** Caught 3 servers on a collision course 8 weeks before breach, provisioned before impact. Zero capacity-related outages in the following 12 months.

---

## End of Day 5 — Wrap-Up Format

```
Day 5 Scorecard
Spaced Repetition: [N/8 Strong] [N Review] [N Weak]
LeetCode: [5 problems — mark each: Strong/Review/Weak]
SQL (Analytical): [Strong/Review/Weak]
Python (ETL): [Strong/Review/Weak]
Tech (Capacity Planning): [Strong/Review/Weak]
Behavioral: [Citi Story #5 — Strong / Needs work]

Notable captures (Sean's words):
→
→